In [ ]:
#delete the following from the sentences:
    #Relative nouns  الذي التي اللذان اللتان اللذين اللتين الذين اللاتي اللواتي اللائي من ما
    #conditional nouns اذا لو لولا كلما إن من ما كيفما اينما
    #passivation مياو
    #declension لا حركات
    #particles (some prepositions and adverbs) احرف الجر 
#convert verbs to their roots
#if verbs and nouns are indicating a female, add the sign بنت
#if dual nouns: add the sign اثنان (two)
#if plural: add the sign كثير 
#past tense: add the sign قبل
#present continuous: add the sign مازال or الان or اثناء
#future: add the sign بعد or قريبا
#command sentences: add the sign لازم
#question :  ? sign in the beginning, question word in the end (من كيف متى اين لماذا)
# لماذا is replaced with سبب
#negation: add لا or لا يوجد or ابدا then remove لم or whatever indicated this sentence has negation
# مرارا وتكرارا : do the verb sign three times (انا كلام كلام كلام)


_______________________________________________________

In [ ]:
#intent detected: translation
#llm model to remove extra words and retrieve the 'to-be-translated' sentence
#order,words with multiple meanings is fixed in the finetuned model

In [ ]:
mle = MLEDisambiguator.pretrained() #mle stands for maximum likelihood estimation
QUESTION_WORDS = {"من", "ماذا", "أين", "اين", "متى", "كيف", "لماذا", "كم", "أي"}
NEG_WORDS = {"لا", "لم", "لن", "ليس", "ما", "أبدًا", "ابدا"}
STOP_WORDS = {"و", "ف", "ثم"}

def transform_to_arsl(sentence):
    repeat_verb = False
    if "مرارا وتكرارا" in sentence:
        repeat_verb = True
        sentence = sentence.replace("مرارا وتكرارا", "")
    
    tokens = simple_word_tokenize(sentence)
    disambig_results = mle.disambiguate(tokens)    
    arsl_sentence = []
    question_word = None
    negation_word = None
    has_q_mark = "؟" in sentence or "?" in sentence
    
    for i, result in enumerate(disambig_results):
        analysis = result.analyses[0].analysis # get the top-ranked analysis
        feat = analysis.features #this dictionary has tags to give the words
        token = tokens[i]
        
        if token in STOP_WORDS: continue
        
        if feat['pos'] in ['pron_rel', 'prep', 'conj']: continue #HANDLE DELETIONS (Particles/Relative Nouns)
        
        if token in QUESTION_WORDS:
            question_word = "سبب" if token == "لماذا" else token
            continue

        if token in NEG_WORDS:
            negation_word = token # Or you can map all to "لا" or "لا يوجد"
            continue

        if feat['gen'] == 'f' and feat['rat'] == 'y': arsl_sentence.append("بنت") # gender (Human Female)
        # if feat['vox'] == p : passivaiton (idk what to do yet)

          
        # Number (Dual/Plural)
        if feat['num'] == 'd': arsl_sentence.append("اثنان")
        elif feat['num'] == 'p': arsl_sentence.append("كثير")
            
        # Tense (Past/Command)
        if feat['asp'] == 'p': arsl_sentence.append("قبل")
        elif feat['asp'] == 'c': arsl_sentence.append("لازم")
        elif feat.get('asp') == 'i': arsl_sentence.append("الآن") # Present Continuous (Imperfective)
        
        if feat.get('prc0') == 'fut_s' or token == "سوف":
            arsl_sentence.append("قريبا")
            if token == "سوف": continue
        
        lemma = dediac_ar(feat.get('lex', token))
            
        arsl_sentence.append(analysis.lemma)
        if repeat_verb and feat['pos'] == 'verb':
            arsl_sentence.extend([lemma, lemma, lemma])
        else:
            arsl_sentence.append(lemma)
    
    if negation_word: arsl_sentence.append("لا")
    if question_word: arsl_sentence.append(question_word)
    if has_q_mark or question_word: arsl_sentence.insert(0, "؟")

    return " ".join(arsl_sentence)

In [ ]:
input = "هل البنتان قرأتا الكتاب مرارا وتكرارا؟"
output = transform_to_arsl(input)
print(output)